In [1]:
from dotenv import load_dotenv    # Import the function that loads variables from the .env file
load_dotenv()                     # Read the .env file and load its variables into the Python environment

True

In [2]:
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv('GROQ_API_KEY'),
    base_url='https://api.groq.com/openai/v1'
)

In [3]:
def llm(prompt):
    response = openai_client.responses.create(
        model='llama-3.1-8b-instant',
        input=prompt
    )
    return response.output_text

In [4]:
llm("Hi, What's up?")

"Not much. It's nice to chat with you. What's on your mind? Do you have any questions, topics you'd like to discuss, or just need someone to talk to?"

In [5]:
question = 'I just discovered the course. Can I join now?'
answer = llm(question)
print(answer)

I'm happy to help, but I need a bit more information. What course are you referring to? Could you please provide more context or details about the course you're interested in joining?


In [6]:
answer = llm(prompt)
print(answer)

NameError: name 'prompt' is not defined

In [7]:
def rag(question):
    search_results = search(question)
    user_prompt = build_prompt(question, search_results)
    return llm(user_prompt)

In [8]:
import requests

docs_url = 'https://datatalks.club/faq/json/courses.json'
response = requests.get(docs_url)
courses_raw = response.json()

In [9]:
courses_raw

[{'course': 'machine-learning-zoomcamp',
  'course_name': 'ML Zoomcamp',
  'path': '/json/machine-learning-zoomcamp.json',
  'questions_count': 472},
 {'course': 'llm-zoomcamp',
  'course_name': 'LLM Zoomcamp',
  'path': '/json/llm-zoomcamp.json',
  'questions_count': 79},
 {'course': 'data-engineering-zoomcamp',
  'course_name': 'Data Engineering Zoomcamp',
  'path': '/json/data-engineering-zoomcamp.json',
  'questions_count': 402},
 {'course': 'mlops-zoomcamp',
  'course_name': 'MLOps Zoomcamp',
  'path': '/json/mlops-zoomcamp.json',
  'questions_count': 255}]

In [10]:
documents = []
url_prefix = 'https://datatalks.club/faq'

for course in courses_raw:
    course_url = f'{url_prefix}{course["path"]}'

    course_response = requests.get(course_url)
    course_response.raise_for_status()
    course_data = course_response.json()

    documents.extend(course_data)

len(documents)

1208

In [11]:
documents[0]

{'id': '0e38656cfb',
 'course': 'machine-learning-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'How do I submit homework?',
 'answer': "- Do the tasks locally\n- Publish your code (e.g., in your own GitHub repo)\n- Submit your answers via the homework form and include the URL to your code\n- You will see the answers only after the deadline\n- Homeworks are in the cohorts folder, e.g. for 2025 it's [`cohorts/2025`](https://github.com/DataTalksClub/machine-learning-zoomcamp/tree/master/cohorts/2025)\n- The forms for submitting the homework are in the [course management platform](https://courses.datatalks.club/)"}

In [12]:
documents[312]

{'id': '2c1cb358cb',
 'course': 'machine-learning-zoomcamp',
 'section': 'Projects (Midterm and Capstone)',
 'question': 'How to conduct peer reviews for projects?',
 'answer': "Previous cohorts' projects page has instructions (YouTube).\n\n[GitHub Instructions](https://github.com/DataTalksClub/machine-learning-zoomcamp/blob/master/cohorts/2022/projects.md#midterm-project)\n\nAlexey and his team will compile a Google Sheet with links to submitted projects using our hashed emails, similar to how we check the leaderboard for homework. These will be our projects to review within the evaluation deadline."}

In [13]:
documents[1207]

{'id': '0d200c8c58',
 'course': 'mlops-zoomcamp',
 'section': 'Capstone Project',
 'question': 'Homework: What is the criteria of scoring home work?',
 'answer': 'Each homework assignment has a scoring system based on the following criteria:\n\n- Answering 6 questions correctly: **6 points**\n- Adding 7 public learning items: **7 points**\n- Adding 1 valid question to the FAQ: **1 point**\n\nIn total, you can earn up to **14 points** per homework, which will contribute to the leaderboard ranking.'}

In [14]:
from minsearch import Index

index = Index(
    text_fields=['question', 'section', 'answer'],
    keyword_fields=['course']
)

index.fit(documents)

In [15]:
search_results = index.search(question,
             boost_dict={'question': 2.0}, 
             filter_dict={'course':"llm-zoomcamp"},
             num_results=5)
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [16]:
def search(question, course='llm-zoomcamp'):
    boost_dict = {'question': 2.0, 'section': 0.5}
    filter_dict = {'course': course}

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5
    )

In [17]:
search_results = search(question)

In [18]:
context = '''
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
'''

In [19]:
INSTRUCTIONS = '''
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."
'''

In [20]:
USER_PROMPT_TEMPLATE = '''
Question:
{question}

Context:
{context}
'''

In [22]:
def build_context(search_results):
    lines = []

    for doc in search_results:
        lines.append(doc['section'])
        lines.append('Q: ' + doc['question'])
        lines.append('A: ' + doc['answer'])
        lines.append('')

    return '\n'.join(lines).strip()

In [23]:
context = build_context(search_results)
print(context)

General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.

You can only peer-review projects at the time the course is run

In [24]:
def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format(
        question=question,
        context=context
    )
    return prompt.strip()

In [25]:
prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

In [26]:
response = openai_client.responses.create(
    model='llama-3.1-8b-instant',
    input=prompt
)

In [27]:
response.output_text

'Yes, you can join the course now. If your goal is to receive a certificate, you need to submit your project while the form is still accepting submissions.'

In [28]:
print(response.model_dump_json(indent=2))

{
  "id": "resp_01krnngcgwezxbwt63cm69khny",
  "created_at": 1778843660.0,
  "error": null,
  "incomplete_details": null,
  "instructions": null,
  "metadata": {},
  "model": "llama-3.1-8b-instant",
  "object": "response",
  "output": [
    {
      "id": "resp_01krnngcgwezxr8djy11r5n2c4",
      "summary": [],
      "type": "reasoning",
      "content": null,
      "encrypted_content": null,
      "status": "completed"
    },
    {
      "id": "msg_01krnngcgwezyahbb87n82fhtc",
      "content": [
        {
          "annotations": [],
          "text": "Yes, you can join the course now. If your goal is to receive a certificate, you need to submit your project while the form is still accepting submissions.",
          "type": "output_text",
          "logprobs": null
        }
      ],
      "role": "assistant",
      "status": "completed",
      "type": "message",
      "phase": null
    }
  ],
  "parallel_tool_calls": true,
  "temperature": 1.0,
  "tool_choice": "auto",
  "tools": [],
 

In [29]:
response.usage

ResponseUsage(input_tokens=366, input_tokens_details=InputTokensDetails(cached_tokens=0), output_tokens=33, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=399)

In [30]:
response.output[0].content[0].text

TypeError: 'NoneType' object is not subscriptable

In [31]:
input_price = 0.75 / 1_000_000
output_price = 4.50 / 1_000_000

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.000423

In [33]:
message_history = [
    {'role': 'developer', 'content': INSTRUCTIONS},
    {'role': 'user', 'content': prompt}
]
response = openai_client.responses.create(
    model='llama-3.1-8b-instant',
    input=prompt
)


In [34]:
response.output_text

'Yes, you can join the course now. However, if you want to receive a certificate, you need to submit your project while submissions are still being accepted.'

In [35]:
def llm(instructions, user_prompt, model='llama-3.1-8b-instant'):
    message_history = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': user_prompt}
    ]

    response = openai_client.responses.create(
        model=model,
        input=message_history
    )

    return response.output_text

In [36]:
def rag(query, model='llama-3.1-8b-instant'):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [37]:
answer = rag(question)
print(answer)

Q: I just discovered the course. Can I still join?

A: Yes, but if you want to receive a certificate, you need to submit your project while the course is still accepting submissions.

Note: If you want to join and receive a certificate, you might want to refer to when the course will be offered next according to this other Q&A: Summer 2025.
